In [ ]:
import pandas as pd 
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#ccba
ccba = pd.read_csv('/content/drive/MyDrive/?????_first/public_train_x_ccba_full_hashed.csv')
#cdtx
cdtx = pd.read_csv('/content/drive/MyDrive/?????_first/public_train_x_cdtx0001_full_hashed.csv')
#custinfo
custinfo = pd.read_csv('/content/drive/MyDrive/?????_first/public_train_x_custinfo_full_hashed.csv')
#dp
dp = pd.read_csv('/content/drive/MyDrive/?????_first/public_train_x_dp_full_hashed.csv')
#remit
remit = pd.read_csv('/content/drive/MyDrive/?????_first/public_train_x_remit1_full_hashed.csv')
#train_alert_time
train_alert_time = pd.read_csv('/content/drive/MyDrive/?????_first/train_x_alert_date.csv')
#predict_alert_time
predict_alert_time = pd.read_csv('/content/drive/MyDrive/?????_first/public_x_alert_date.csv')
#y
y = pd.read_csv('/content/drive/MyDrive/?????_first/train_y_answer.csv')
#????
doc = pd.read_csv('/content/drive/MyDrive/?????_first/??????????????.csv')
#prev_list set to 0 before submission
doc_merged = custinfo.merge(doc, on='alert_key', how='right').merge(predict_alert_time, on='alert_key', how='left')
doc_merged = doc_merged[~doc_merged['cust_id'].isnull()]
max_doc = doc_merged.groupby('cust_id')['date'].max().reset_index()
max_doc.rename(columns={'date':'max_date'},inplace=True)
doc_merged = doc_merged.merge(max_doc, on='cust_id', how='left')
prev_list = doc_merged[doc_merged['max_date']>doc_merged['date']]['alert_key'].tolist()

In [ ]:
def train_alert_process_func(data, custinfo, predict_alert_time):
  alert_data = data[data['cust_id'].isin(custinfo[custinfo['alert_key'].isin(predict_alert_time['alert_key'].tolist())]['cust_id'].tolist())]
  train_data = data[~data['cust_id'].isin(custinfo[custinfo['alert_key'].isin(predict_alert_time['alert_key'].tolist())]['cust_id'].tolist())]
  train_data['y'] = 0
  sar_idx = train_data[train_data['cust_id'].isin(custinfo[custinfo['alert_key'].isin(y[y['sar_flag']==1]['alert_key'].tolist())]['cust_id'].tolist())].index
  train_data.loc[sar_idx, 'y'] = 1
  return train_data, alert_data

In [ ]:
#Data Cleansing
#SAR????X?
def train_prev_d(x, day):
  prev_d = x.groupby('cust_id')['tx_date'].max() - day
  prev_d = prev_d.reset_index()
  prev_d.rename(columns={'tx_date':'prev_d'}, inplace=True)
  x = x.merge(prev_d, on='cust_id', how='left')
  x.drop(x[x['tx_date']<x['prev_d']].index,inplace=True)
  x.pop('prev_d')
  return x
#?????
def preprocess(data):
  dict1 = {}
  idx = 0
  num = 0
  for i in range(0,395,1):
    dict1[i] = str(idx)
    num += 1
    if num == 7:
      idx += 1
      num = 0
  data['tx_date_group'] = data.tx_date.map(lambda x: dict1[x])
  data['session_cust_id'] = data.tx_date_group + data.cust_id
  data['date_cust_id'] = data.tx_date.astype(str) + data.cust_id
  data['date_time_cust_id'] = data.tx_date.astype(str) + data.tx_date.astype(str) + data.cust_id
  data['tx_year'] = 2021
  data['tx_date_formal'] = data['tx_date']+1
  data.loc[data[data['tx_date_formal']>365].index,'tx_year'] = 2022
  data.loc[data[data['tx_date_formal']>365].index, 'tx_date_formal'] = \
  data.loc[data[data['tx_date_formal']>365].index, 'tx_date_formal'] -365
  data["tx_date_formal"] = data["tx_year"]*1000 + data["tx_date_formal"]
  data['tx_date_formal'] = pd.to_datetime(data['tx_date_formal'], format="%Y%j")
  return data

In [ ]:
def cdtx_feature_func(data):
  ##(t-1)d diff
  #data['prev_d_diff'] = data['trans_date'] - data.groupby('cust_id')['trans_date'].shift(1)
  #data['prev_d_diff'].fillna(15, inplace=True)
  #amt
  all_amt_sum = (data.groupby('cust_id')['amt'].sum()/data.groupby('cust_id')['amt'].sum().max())\
              .rename('all_amt_sum').reset_index()
  data = data.merge(all_amt_sum, on='cust_id', how='left')
  date_amt_sum = (data.groupby(['cust_id','date'])['amt'].sum()/data.groupby(['cust_id','date'])['amt'].sum().max())\
              .rename('date_amt_sum').reset_index()
  data = data.merge(date_amt_sum, on=['cust_id','date'], how='left')
  #txn_cnt
  all_txn_cnt = data.groupby('cust_id')['date'].count().rename('cdtx_all_txn_cnt').reset_index()
  data = data.merge(all_txn_cnt, on='cust_id', how='left')
  data['cdtx_all_txn_cnt'].fillna(0, inplace=True)
  date_txn_cnt = data.groupby(['cust_id','date'])['amt'].count().rename('date_txn_cnt').reset_index()
  data = data.merge(date_txn_cnt, on=['cust_id','date'], how='left')
  #country
  all_country_cnt = data.groupby('cust_id')['country'].count().rename('cdtx_all_country_cnt').reset_index()
  data = data.merge(all_country_cnt, on='cust_id', how='left')
  data['cdtx_all_country_cnt'].fillna(0, inplace=True)
  date_country_cnt = data.groupby(['cust_id','date'])['country'].count().rename('date_country_cnt').reset_index()
  data = data.merge(date_country_cnt, on=['cust_id','date'], how='left')
  #cur_type
  all_cur_type_cnt = data.groupby('cust_id')['cur_type'].count().rename('cdtx_all_cur_type_cnt').reset_index()
  data = data.merge(all_cur_type_cnt, on='cust_id', how='left')
  data['cdtx_all_cur_type_cnt'].fillna(0, inplace=True)
  date_cur_type_cnt = data.groupby(['cust_id','date'])['cur_type'].count().rename('date_cur_type_cnt').reset_index()
  data = data.merge(date_cur_type_cnt, on=['cust_id','date'], how='left')
  #max_min_date_diff
  max_min_date_diff = data.groupby(['cust_id']).apply(lambda x: x['date'].max() - x['date'].min()).rename('max_min_date_diff').reset_index()
  data = data.merge(max_min_date_diff, on='cust_id', how='left')
  return data

In [ ]:
def remit_feature_func(data):
  ##(t-1)d diff
  #data['prev_d_diff'] = data['trans_date'] - data.groupby('cust_id')['trans_date'].shift(1)
  #data['prev_d_diff'].fillna(15, inplace=True)
  #amt
  all_amt_sum = (data.groupby('cust_id')['trade_amount_usd'].sum()/data.groupby('cust_id')['trade_amount_usd'].sum().max())\
              .rename('all_amt_sum').reset_index()
  data = data.merge(all_amt_sum, on='cust_id', how='left')
  time_amt_sum = (data.groupby(['cust_id','trans_date'])['trade_amount_usd'].sum()/data.groupby(['cust_id','trans_date'])['trade_amount_usd'].sum().max())\
              .rename('time_amt_sum').reset_index()
  data = data.merge(time_amt_sum, on=['cust_id','trans_date'], how='left')
  data['trans_date'] = data['trans_date']/data['trans_date'].max()
  #txn_cnt
  all_txn_cnt = data.groupby('cust_id')['trans_date'].count().rename('remit_all_txn_cnt').reset_index()
  data = data.merge(all_txn_cnt, on='cust_id', how='left')
  data['remit_all_txn_cnt'].fillna(0, inplace=True)
  date_txn_cnt = data.groupby(['cust_id','trans_date'])['trans_no'].count().rename('date_txn_cnt').reset_index()
  data = data.merge(date_txn_cnt, on=['cust_id','trans_date'], how='left')
  #max_min_date_diff
  max_min_date_diff = data.groupby(['cust_id']).apply(lambda x: x['trans_date'].max() - x['trans_date'].min()).rename('max_min_date_diff').reset_index()
  data = data.merge(max_min_date_diff, on='cust_id', how='left')
  remit_pivot = data.groupby(['cust_id','trans_date','trans_no'])['trade_amount_usd'].sum().reset_index().pivot_table(index=['cust_id','trans_date'],columns='trans_no',values='trade_amount_usd').reset_index()
  remit_pivot.fillna(0, inplace=True)
  remit_col = list(remit_pivot.columns)
  remit_col.remove('cust_id')
  remit_col.remove('trans_date')
  remit_pivot['remit_txn_sum'] = remit_pivot[remit_col].sum(axis=1)
  for col in remit_col:
    remit_pivot[col] = remit_pivot[col]/remit_pivot['remit_txn_sum']
  for col in list(set([num for num in range(0,5,1)]) -  set(remit_pivot.columns[2:])):
    remit_pivot[col] = 0.0
  remit_pivot.rename(columns={0:'remit_trans_no_0',1:'remit_trans_no_1',2:'remit_trans_no_2',3:'remit_trans_no_3',4:'remit_trans_no_4'}, inplace=True)
  remit_pivot.fillna(0, inplace=True)
  data = data.merge(remit_pivot, on=['cust_id', 'trans_date'], how='left')
  return data

In [ ]:
#session ??????
def dp_feature_func(data):
  #session ??????
  session_amt_diff = data.groupby(['session_cust_id','debit_credit'])['tx_amt'].sum().reset_index()
  session_amt_diff = pd.pivot_table(session_amt_diff, index='session_cust_id', columns='debit_credit', values='tx_amt')
  session_amt_diff.fillna(1, inplace=True)
  session_amt_diff['session_amt_diff_ratio'] = \
    abs(session_amt_diff['CR'] - session_amt_diff['DB']) / abs(session_amt_diff['CR'] + session_amt_diff['DB'])
  session_amt_diff = session_amt_diff.reset_index()[['session_cust_id','session_amt_diff_ratio']]
  data = data.merge(session_amt_diff, on='session_cust_id', how='left')
  #ATM session ??????
  atm_session_amt_diff = data[data['ATM']==1].groupby(['session_cust_id','debit_credit'])['tx_amt'].sum().reset_index()
  atm_session_amt_diff = pd.pivot_table(atm_session_amt_diff, index='session_cust_id', columns='debit_credit', values='tx_amt')
  atm_session_amt_diff.fillna(1, inplace=True)
  atm_session_amt_diff['atm_session_amt_diff_ratio'] = \
    abs(atm_session_amt_diff['CR'] - atm_session_amt_diff['DB']) / abs(atm_session_amt_diff['CR'] + atm_session_amt_diff['DB'])
  atm_session_amt_diff = atm_session_amt_diff.reset_index()[['session_cust_id','atm_session_amt_diff_ratio']]
  data = data.merge(atm_session_amt_diff, on='session_cust_id', how='left')
  #?? ??????
  date_amt_diff = data.groupby(['date_cust_id','debit_credit'])['tx_amt'].sum().reset_index()
  date_amt_diff = pd.pivot_table(date_amt_diff, index='date_cust_id', columns='debit_credit', values='tx_amt')
  date_amt_diff.fillna(1, inplace=True)
  date_amt_diff['date_amt_diff_ratio'] = \
  abs(date_amt_diff['CR'] - date_amt_diff['DB']) / abs(date_amt_diff['CR'] + date_amt_diff['DB'])
  date_amt_diff = date_amt_diff.reset_index()[['date_cust_id','date_amt_diff_ratio']]
  data = data.merge(date_amt_diff, on=['date_cust_id'], how='left')
  #ATM?? ??????
  atm_date_amt_diff = data[data['ATM']==1].groupby(['date_cust_id','debit_credit'])['tx_amt'].sum().reset_index()
  atm_date_amt_diff = pd.pivot_table(atm_date_amt_diff, index='date_cust_id', columns='debit_credit', values='tx_amt')
  atm_date_amt_diff.fillna(1, inplace=True)
  atm_date_amt_diff['atm_date_amt_diff_ratio'] = \
  abs(atm_date_amt_diff['CR'] - atm_date_amt_diff['DB']) / abs(atm_date_amt_diff['CR'] + atm_date_amt_diff['DB'])
  atm_date_amt_diff = atm_date_amt_diff.reset_index()[['date_cust_id','atm_date_amt_diff_ratio']]
  data = data.merge(atm_date_amt_diff, on=['date_cust_id'], how='left')
  #?? ??????
  date_time_amt_diff = data.groupby(['date_time_cust_id','debit_credit'])['tx_amt'].sum().reset_index()
  date_time_amt_diff = pd.pivot_table(date_time_amt_diff, index='date_time_cust_id', columns='debit_credit', values='tx_amt')
  date_time_amt_diff.fillna(1, inplace=True)
  date_time_amt_diff['date_time_amt_diff_ratio'] = \
  abs(date_time_amt_diff['CR'] - date_time_amt_diff['DB']) / abs(date_time_amt_diff['CR'] + date_time_amt_diff['DB'])
  date_time_amt_diff = date_time_amt_diff.reset_index()[['date_time_cust_id','date_time_amt_diff_ratio']]
  data = data.merge(date_time_amt_diff, on=['date_time_cust_id'], how='left')
  #ATM?? ??????
  atm_date_time_amt_diff = data[data['ATM']==1].groupby(['date_time_cust_id','debit_credit'])['tx_amt'].sum().reset_index()
  atm_date_time_amt_diff = pd.pivot_table(atm_date_time_amt_diff, index='date_time_cust_id', columns='debit_credit', values='tx_amt')
  atm_date_time_amt_diff.fillna(1, inplace=True)
  atm_date_time_amt_diff['atm_date_time_amt_diff_ratio'] = \
  abs(atm_date_time_amt_diff['CR'] - atm_date_time_amt_diff['DB']) / abs(atm_date_time_amt_diff['CR'] + atm_date_time_amt_diff['DB'])
  atm_date_time_amt_diff = atm_date_time_amt_diff.reset_index()[['date_time_cust_id','atm_date_time_amt_diff_ratio']]
  data = data.merge(atm_date_time_amt_diff, on=['date_time_cust_id'], how='left')
  #?????? tx_cnt_date_time
  tx_cnt_date_time = data.groupby(['cust_id','tx_date','tx_time'])['debit_credit'].count().reset_index()
  tx_cnt_date_time.rename(columns={'debit_credit':'tx_cnt_date_time'}, inplace=True)
  data = data.merge(tx_cnt_date_time, on=['cust_id','tx_date','tx_time'], how='left')
  #?????? tx_cnt_date
  tx_cnt_date = data.groupby(['cust_id','tx_date'])['debit_credit'].count().reset_index()
  tx_cnt_date.rename(columns={'debit_credit':'tx_cnt_date'}, inplace=True)
  data = data.merge(tx_cnt_date, on=['cust_id','tx_date'], how='left')
  #?????? txbranch_day_cnt
  txbranch_day_time_cnt = data.groupby(['cust_id','tx_date','tx_time'])['txbranch'].count().reset_index()
  txbranch_day_time_cnt.rename(columns={'txbranch':'txbranch_day_time_cnt'}, inplace=True)
  data = data.merge(txbranch_day_time_cnt, on=['cust_id','tx_date','tx_time'], how='left')
  #?????? txbranch_day_cnt
  txbranch_day_cnt = data.groupby(['cust_id','tx_date'])['txbranch'].count().reset_index()
  txbranch_day_cnt.rename(columns={'txbranch':'txbranch_day_cnt'}, inplace=True)
  data = data.merge(txbranch_day_cnt, on=['cust_id','tx_date'], how='left')
  #??ATM ??????
  day_atm_txn_ratio = data.groupby(['cust_id','tx_date'])['ATM'].sum().reset_index()
  day_atm_txn_ratio.rename(columns={'ATM':'day_atm_txn_ratio'}, inplace=True)
  data = data.merge(day_atm_txn_ratio, on=['cust_id','tx_date'], how='left')
  data.day_atm_txn_ratio = data.day_atm_txn_ratio / data.tx_cnt_date
  #??ATM ??????
  day_time_atm_txn_ratio = data.groupby(['cust_id','tx_date','tx_time'])['ATM'].sum().reset_index()
  day_time_atm_txn_ratio.rename(columns={'ATM':'day_time_atm_txn_ratio'}, inplace=True)
  data = data.merge(day_time_atm_txn_ratio, on=['cust_id','tx_date','tx_time'], how='left')
  data.day_time_atm_txn_ratio = data.day_time_atm_txn_ratio / data.tx_cnt_date_time
  #???? ??????
  day_cross_bank_ratio = data.groupby(['cust_id','tx_date'])['cross_bank'].sum().reset_index()
  day_cross_bank_ratio.rename(columns={'cross_bank':'day_cross_bank_ratio'}, inplace=True)
  data = data.merge(day_cross_bank_ratio, on=['cust_id','tx_date'], how='left')
  data.day_cross_bank_ratio = data.day_cross_bank_ratio / data.tx_cnt_date
  #???? ??????
  day_time_cross_bank_ratio = data.groupby(['cust_id','tx_date','tx_time'])['cross_bank'].sum().reset_index()
  day_time_cross_bank_ratio.rename(columns={'cross_bank':'day_time_cross_bank_ratio'}, inplace=True)
  data = data.merge(day_time_cross_bank_ratio, on=['cust_id','tx_date','tx_time'], how='left')
  data.day_time_cross_bank_ratio = data.day_time_cross_bank_ratio / data.tx_cnt_date_time

  ##
  #?session???? tx_cnt_session
  tx_cnt_session = data.groupby(['session_cust_id'])['debit_credit'].count().reset_index()
  tx_cnt_session.rename(columns={'debit_credit':'tx_cnt_session'}, inplace=True)
  data = data.merge(tx_cnt_session, on=['session_cust_id'], how='left')
  #?session???? txbranch_session_cnt
  txbranch_session_cnt = data.groupby(['session_cust_id'])['txbranch'].count().reset_index()
  txbranch_session_cnt.rename(columns={'txbranch':'txbranch_session_cnt'}, inplace=True)
  data = data.merge(txbranch_session_cnt, on=['session_cust_id'], how='left')
  #?session?? ??????
  session_cross_bank_ratio = data.groupby(['session_cust_id'])['cross_bank'].sum().reset_index()
  session_cross_bank_ratio.rename(columns={'cross_bank':'session_cross_bank_ratio'}, inplace=True)
  data = data.merge(session_cross_bank_ratio, on=['session_cust_id'], how='left')
  data.session_cross_bank_ratio = data.session_cross_bank_ratio / data.tx_cnt_date
  #?sessionATM ??????
  session_atm_txn_ratio = data.groupby(['session_cust_id'])['ATM'].sum().reset_index()
  session_atm_txn_ratio.rename(columns={'ATM':'session_atm_txn_ratio'}, inplace=True)
  data = data.merge(session_atm_txn_ratio, on=['session_cust_id'], how='left')
  data.session_atm_txn_ratio = data.session_atm_txn_ratio / data.tx_cnt_date
  #time_diff
  distinct_date_time = data[['cust_id','tx_date','tx_time']].drop_duplicates().sort_values(['cust_id','tx_date','tx_time']).reset_index(drop=True)
  distinct_date_time['date_diff'] = distinct_date_time.groupby('cust_id').apply(lambda x: x['tx_date'] - x['tx_date'].shift(1)).reset_index(drop=True)
  distinct_date_time['time_diff'] = distinct_date_time.groupby('cust_id').apply(lambda x: x['tx_time'] - x['tx_time'].shift(1)).reset_index(drop=True)
  distinct_date_time.fillna(0, inplace=True)
  distinct_date_time['time_diff'] = (distinct_date_time['date_diff']*24) + distinct_date_time['time_diff']
  data = data.merge(distinct_date_time[['cust_id', 'tx_date', 'tx_time', 'time_diff']], on=['cust_id', 'tx_date', 'tx_time'], how='left')

  return data

In [ ]:
def model_training_1(data, columns):
  #??????
  Y = data['y']
  X = data[columns[1:-1]]
  test_size = 0.2
  seed = 42
#   X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=test_size, random_state=seed)#, stratify = Y)
  model = XGBClassifier(
                    base_score= 0.5, 
                booster= 'gbtree', 
                colsample_bylevel= 1, 
                colsample_bynode= 1, 
                colsample_bytree= 1, 
                gamma= 0, 
                learning_rate= 0.03,
                max_delta_step= 0, 
                max_depth= 2, 
                min_child_weight= 1, 
#                 missing= None, 
                n_estimators= 90, 
                nthread= 16, 
                objective= 'binary:logistic', 
                reg_alpha= 0, 
                reg_lambda= 1, 
                scale_pos_weight= 1, 
                seed= 0, 
                subsample= 0.9,
                verbosity= 1
  )
  model.fit(X, Y)
#   y_pred = model.predict(X_test)
#   predictions = [round(value) for value in y_pred]
  # evaluate predictions
#   accuracy = accuracy_score(y_test, predictions)
#   print("Accuracy: %.2f%%" % (accuracy * 100.0))
#   f1 = f1_score(y_test, predictions)
#   print("F1 Scoure: %.2f%%" % (f1 * 100.0))
#   print(precision_score(y_test, predictions))
#   print(recall_score(y_test, predictions))
  feature_importance = pd.DataFrame({'columns':list(X.columns),'score':model.feature_importances_})
  return X, model, feature_importance

In [ ]:
#??model training 1????
#predict_proba?0.05?????????????
def mapping(proba):
    if proba <= 0.1:
        return 1
    elif 0.1 < proba <= 0.2:
        return 2
    elif 0.2 < proba <= 0.3:
        return 3
    elif 0.3 < proba <= 0.4:
        return 4
    elif 0.4 < proba <= 0.5:
        return 5
    elif 0.5 < proba <= 0.6:
        return 6
    elif 0.6 < proba <= 0.7:
        return 7
    elif 0.7 < proba <= 0.8:
        return 8
    elif 0.8 < proba <= 0.9:
        return 9
    elif 0.9 < proba <= 1:
        return 10
def debit_credit_ratio_func(data):
  #id
  debit_credit_ratio = data.groupby(['cust_id'])['debit_credit'].value_counts().rename('debit_credit_ratio').reset_index()
  debit_credit_ratio = debit_credit_ratio.pivot_table(values='debit_credit_ratio', index=['cust_id'], columns='debit_credit')
  debit_credit_ratio.fillna(0, inplace=True)
  debit_credit_ratio['debit_credit_ratio'] = debit_credit_ratio['DB']/debit_credit_ratio.sum(axis=1)
  debit_credit_ratio = debit_credit_ratio.reset_index()[['cust_id','debit_credit_ratio']]
  data = data.merge(debit_credit_ratio, on=['cust_id'], how='left')
  return debit_credit_ratio
def atm_debit_credit_ratio_func(data):
  #id
  atm_debit_credit_ratio = data[data['ATM']==1].groupby(['cust_id'])['debit_credit'].value_counts().rename('atm_debit_credit_ratio').reset_index()
  atm_debit_credit_ratio = atm_debit_credit_ratio.pivot_table(values='atm_debit_credit_ratio', index=['cust_id'], columns='debit_credit')
  atm_debit_credit_ratio.fillna(0, inplace=True)
  atm_debit_credit_ratio['atm_debit_credit_ratio'] = atm_debit_credit_ratio['DB']/atm_debit_credit_ratio.sum(axis=1)
  atm_debit_credit_ratio = atm_debit_credit_ratio.reset_index()[['cust_id','atm_debit_credit_ratio']]
  data = data.merge(atm_debit_credit_ratio, on=['cust_id'], how='left')
  return atm_debit_credit_ratio
def txbranch_debit_credit_ratio_func(data):
  txbranch_debit_credit_ratio = data[(~data['txbranch'].isnull())].groupby(['cust_id'])['debit_credit'].value_counts().rename('txbranch_debit_credit_ratio').reset_index()
  txbranch_debit_credit_ratio = txbranch_debit_credit_ratio.pivot_table(values='txbranch_debit_credit_ratio', index=['cust_id'], columns='debit_credit')
  txbranch_debit_credit_ratio.fillna(0, inplace=True)
  txbranch_debit_credit_ratio['txbranch_debit_credit_ratio'] = txbranch_debit_credit_ratio['DB']/txbranch_debit_credit_ratio.sum(axis=1)
  txbranch_debit_credit_ratio = txbranch_debit_credit_ratio.reset_index()[['cust_id','txbranch_debit_credit_ratio']]
  data = data.merge(txbranch_debit_credit_ratio, on=['cust_id'], how='left')
  return txbranch_debit_credit_ratio
def all_txn_cnt(data):
  all_txn_cnt = data.groupby('cust_id')['tx_date'].count().rename('all_txn_cnt').reset_index()
  data = data.merge(all_txn_cnt, on='cust_id', how='left')
  return all_txn_cnt
def result_preprocess_func(data, X, model):
  data_db_cr_ratio = debit_credit_ratio_func(data)
  #atm_data_db_cr_ratio = atm_debit_credit_ratio_func(data)
  txbranch_debit_credit_ratio = txbranch_debit_credit_ratio_func(data)
  data_all_txn_cnt = all_txn_cnt(data)
  data['proba'] = model.predict_proba(X)[:,1]
  result = data[['cust_id','proba','y']]
  result['level'] = result["proba"].map(mapping)
  result = result[['cust_id', 'level', 'y']]
  result = result.groupby(['cust_id','level']).count().reset_index().pivot_table(index='cust_id', columns='level', values='y')
  result.fillna(0, inplace=True)
  result = result.div(result.sum(axis=1), axis=0).reset_index()
  for col in list(set([num for num in range(1,11,1)]) -  set(result.columns[1:])):
    result[col] = 0.0
  result = result[['cust_id', 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]]
  result = result.merge(data[['cust_id','y']].drop_duplicates(), on='cust_id', how='left')
  result = result.merge(data_db_cr_ratio, on='cust_id', how='left')
  #result = result.merge(atm_data_db_cr_ratio, on='cust_id', how='left')
  result = result.merge(txbranch_debit_credit_ratio, on='cust_id', how='left')
  result = result.merge(data_all_txn_cnt, on='cust_id', how='left')
  return result

In [ ]:
#model training 2
#???????SAR
def model_training_2(result):
  result_col = list(result.columns)
  result_col.remove('cust_id')
  result_col.remove('y')
  test_size = 0.2
  seed = 42
#   X_train, X_test, y_train, y_test = train_test_split(result[result_col], result['y'], test_size=test_size, random_state=seed)#, stratify = result['y'])
  model = XGBClassifier(
                    base_score= 0.5, 
                booster= 'gbtree', 
                colsample_bylevel= 1, 
                colsample_bynode= 1, 
                colsample_bytree= 1, 
                gamma= 0, 
                learning_rate= 0.2,
                max_delta_step= 0, 
                max_depth= 2, 
                min_child_weight= 1, 
#                 missing= None, 
                n_estimators=80, 
                nthread= 16, 
                objective= 'binary:logistic', 
                reg_alpha= 0.1, 
                reg_lambda= 1, 
                scale_pos_weight= 1, 
                seed= 0, 
                subsample= 1,
                verbosity= 1
  )
  model.fit(result[result_col], result['y'])
#   y_pred = model.predict(X_test)
#   predictions = [round(value) for value in y_pred]
#   # evaluate predictions
#   accuracy = accuracy_score(y_test, predictions)
#   print("Accuracy: %.2f%%" % (accuracy * 100.0))
#   f1 = f1_score(y_test, predictions)
#   print("F1 Scoure: %.2f%%" % (f1 * 100.0))
#   print(precision_score(y_test, predictions))
#   print(recall_score(y_test, predictions))
  feature_importance = pd.DataFrame({'columns':list(result_col),'score':model.feature_importances_})
  return model, feature_importance, result_col

In [ ]:
def alert_output(alert, model_1, model_2, alert_col, result_col, doc):
  data_db_cr_ratio = debit_credit_ratio_func(alert)
  #atm_db_cr_ratio = atm_debit_credit_ratio_func(alert)
  txbranch_debit_credit_ratio = txbranch_debit_credit_ratio_func(alert)
  data_all_txn_cnt = all_txn_cnt(alert)

  #Remit Submission
  alert_data = alert
  alert_data['proba'] = model_1.predict_proba(alert[alert_col])[:,1]
  alert_result = alert_data[['cust_id','proba']]
  alert_result['level'] = alert_result["proba"].map(mapping)
  alert_result = alert_result[['cust_id', 'level']]
  alert_result['cnt'] = 1
  alert_result = alert_result.groupby(['cust_id','level'])['cnt'].count().reset_index().pivot_table(index='cust_id', columns='level', values='cnt')
  alert_result.fillna(0, inplace=True)
  alert_result = alert_result.div(alert_result.sum(axis=1), axis=0)
  alert_result = alert_result.reset_index()
  for col in list(set([num for num in range(1,11,1)]) -  set(alert_result.columns[1:])):
    alert_result[col] = 0.0
  alert_result = alert_result.merge(data_db_cr_ratio, on='cust_id', how='left')
  #alert_result = alert_result.merge(atm_db_cr_ratio, on='cust_id', how='left')
  alert_result = alert_result.merge(txbranch_debit_credit_ratio, on='cust_id', how='left')
  alert_result = alert_result.merge(data_all_txn_cnt, on='cust_id', how='left')
  alert_pred = model_2.predict_proba(alert_result[result_col])
  # evaluate predictions
  alert_result = alert_result[['cust_id']]
  alert_result['probability'] = alert_pred[:,1]

  final = predict_alert_time.merge(custinfo[['alert_key', 'cust_id']].merge(alert_result, on='cust_id'), on='alert_key', how='left')[['alert_key', 'probability']]
  doc = doc[['alert_key']]
  final = doc.merge(final, on='alert_key', how='left')
  final.fillna(0,inplace=True)
  final.loc[final[final['alert_key'].isin(prev_list)].index,'probability']=0
  return final, alert_result

In [ ]:
def prev_7d_feature_func(data):
  #last 7 days processing
  data_distinct = data[['cust_id', 'tx_date_formal','tx_date']].drop_duplicates().reset_index(drop=True)
  cross_bank_sum = data.groupby(['cust_id','tx_date_formal'])['cross_bank'].sum().rename('cross_bank_sum').reset_index()
  prev_7d_data = data_distinct.merge(cross_bank_sum, on=['cust_id','tx_date_formal'], how='left')
  dbcr_amt_sum = data.groupby(['cust_id','tx_date_formal','debit_credit'])['tx_amt'].sum().rename('debit_credit_sum').reset_index()
  dbcr_amt_sum = pd.pivot_table(dbcr_amt_sum, index=['cust_id','tx_date_formal'], columns='debit_credit', values='debit_credit_sum')
  prev_7d_data = prev_7d_data.merge(dbcr_amt_sum, on=['cust_id','tx_date_formal'], how='left')
  tx_date_sum = data.groupby(['cust_id','tx_date_formal'])['tx_amt'].sum().rename('tx_amt_sum')
  prev_7d_data = prev_7d_data.merge(tx_date_sum, on=['cust_id','tx_date_formal'], how='left')
  tx_cnt = data.groupby(['cust_id','tx_date_formal'])['tx_amt'].count().rename('tx_cnt').reset_index()
  prev_7d_data = prev_7d_data.merge(tx_cnt, on=['cust_id','tx_date_formal'], how='left')
  #??ATM ??????
  day_atm_cnt = data.groupby(['cust_id','tx_date_formal'])['ATM'].sum().rename('day_atm_cnt').reset_index()
  prev_7d_data = prev_7d_data.merge(day_atm_cnt, on=['cust_id','tx_date_formal'], how='left')
  txbranch_day_cnt = data.groupby(['cust_id','tx_date_formal'])['txbranch'].count().rename('txbranch_day_cnt').reset_index()
  prev_7d_data = prev_7d_data.merge(txbranch_day_cnt, on=['cust_id','tx_date_formal'], how='left')
  prev_7d_data.fillna(0, inplace=True)
  prev_7d_data = prev_7d_data.sort_values(['cust_id','tx_date_formal'])
  prev_7d_data = prev_7d_data.set_index('tx_date_formal')
  #feature engineering
  prev_7d_cross_bank_sum = prev_7d_data.groupby('cust_id')['cross_bank_sum'].rolling(window='7D').sum().rename('prev_7d_cross_bank_sum').reset_index()
  prev_7d_CR_sum = prev_7d_data.groupby('cust_id')['CR'].rolling(window='7D').sum().rename('prev_7d_CR_sum').reset_index()
  prev_7d_DB_sum = prev_7d_data.groupby('cust_id')['DB'].rolling(window='7D').sum().rename('prev_7d_DB_sum').reset_index()
  prev_7d_tx_cnt = prev_7d_data.groupby('cust_id')['tx_cnt'].rolling(window='7D').sum().rename('prev_7d_tx_cnt').reset_index()
  prev_7d_txbranch_cnt = prev_7d_data.groupby('cust_id')['txbranch_day_cnt'].rolling(window='7D').sum().rename('prev_7d_txbranch_cnt').reset_index()
  prev_7d_atm_cnt = prev_7d_data.groupby('cust_id')['day_atm_cnt'].rolling(window='7D').sum().rename('prev_7d_atm_cnt').reset_index()

  data_distinct = data_distinct.merge(prev_7d_CR_sum, on=['cust_id','tx_date_formal']).merge(prev_7d_DB_sum, on=['cust_id','tx_date_formal'])\
    .merge(prev_7d_tx_cnt, on=['cust_id','tx_date_formal']).merge(prev_7d_txbranch_cnt, on=['cust_id','tx_date_formal']).merge(prev_7d_atm_cnt, on=['cust_id','tx_date_formal'])\
    .merge(prev_7d_cross_bank_sum, on=['cust_id','tx_date_formal'])

  data_distinct['prev_7d_txbranch_ratio'] = data_distinct.prev_7d_txbranch_cnt / data_distinct.prev_7d_tx_cnt
  data_distinct['prev_7d_atm_ratio'] = data_distinct.prev_7d_atm_cnt / data_distinct.prev_7d_tx_cnt
  data_distinct['prev_7d_cross_bank_ratio'] = data_distinct.prev_7d_cross_bank_sum / data_distinct.prev_7d_tx_cnt
  data_distinct['prev7d_amt_diff_ratio'] = \
    abs(data_distinct['prev_7d_CR_sum'] - data_distinct['prev_7d_DB_sum']) / abs(data_distinct['prev_7d_CR_sum'] + data_distinct['prev_7d_DB_sum'])
  data_distinct = data_distinct[['cust_id','tx_date','prev_7d_tx_cnt','prev7d_amt_diff_ratio','prev_7d_txbranch_ratio','prev_7d_atm_cnt','prev_7d_atm_ratio',
                                 'prev_7d_txbranch_cnt','prev_7d_cross_bank_ratio','prev_7d_cross_bank_sum']]
  data = data.merge(data_distinct, on=['cust_id', 'tx_date'], how='left')
  return data, data_distinct

In [ ]:
#dp zone
train_dp, alert_dp = train_alert_process_func(dp, custinfo, predict_alert_time)
train_dp = train_prev_d(train_dp, 30)
alert_dp = train_prev_d(alert_dp, 30)
train_dp = preprocess(train_dp)
alert_dp = preprocess(alert_dp)
##

train_dp, train_dp_distinct = prev_7d_feature_func(train_dp)
alert_dp, alert_dp_distinct = prev_7d_feature_func(alert_dp)
##
train_dp = dp_feature_func(train_dp)
alert_dp = dp_feature_func(alert_dp)

dp_col = ['cust_id','session_amt_diff_ratio', 
          'date_time_amt_diff_ratio','tx_cnt_date_time','txbranch_day_time_cnt',
          'tx_cnt_date','txbranch_day_cnt','day_atm_txn_ratio',
          'y']
dp_X, dp_model_1, dp_feature_importance_1 = model_training_1(train_dp, dp_col)
dp_result = result_preprocess_func(train_dp, dp_X, dp_model_1)
dp_model_2, dp_feature_importance_2, dp_result_col = model_training_2(dp_result)
dp_col.remove('y')
dp_col.remove('cust_id')
dp_final, alert_dp_result = alert_output(alert_dp, dp_model_1, dp_model_2, dp_col, dp_result_col, doc)
dp_final
#F1 Scoure: 66.66%
#0.9980544747081712
#0.5003657644476956
#F1 Scoure: 28.57%
#0.875
#0.17073170731707318

In [ ]:
dp_feature_importance_1

In [ ]:
dp_feature_importance_2

In [ ]:
dp_final.to_csv('/content/drive/MyDrive/?????_first/submission.csv')